In [1]:
import sys
import numpy as np

sys.path.append("../../../")
from Rain.Rain import Rain
sys.path.pop()

from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout
import tensorflow as tf

2023-07-03 21:02:12.390452: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-07-03 21:02:13.549452: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
import os

def clean():
    folder_paths = ["logs", "../../../data/coord/", "../../../data/divider/", "../../../data/worker/"]  # Replace with the folder path you want to delete files from
    file_extensions = [".npy", ".pkl", ".log"]  # Replace with the file extension you want to delete
    for folder_path in folder_paths:
        if os.path.exists(folder_path):
            for filename in os.listdir(folder_path):
                for file_extension in file_extensions:
                    if filename.endswith(file_extension):
                        file_path = os.path.join(folder_path, filename)
                        os.remove(file_path)
clean()

In [3]:
config = {
  "mode": "local",
  "partitions": 3,
  "num_of_workers": 3,
  "iterations": 3,
  "learning_type": "DL",
  "DL": {
    "lib": {
      "type": "tensorflow",
      "params": {
        "loss": tf.keras.losses.CategoricalCrossentropy(),
        "optimizer": tf.keras.optimizers.Adam(learning_rate=0.001),

      }
    },
    "lr": 0.001,
    "epochs": 2,
    "batch_size": 128,
  }
}

In [4]:
def get_train_data():
    return np.load("../../../data/MNIST/train_data.npy"), np.load(
        "../../../data/MNIST/train_labels.npy"
    )


def get_test_data():
    return np.load("../../../data/MNIST/test_data.npy"), np.load(
        "../../../data/MNIST/test_labels.npy"
    )


def partition_train_data(X_train, y_train, partitions):
    num_samples = X_train.shape[0]

    # Create an array of indices from 0 to num_samples - 1
    indices = np.arange(num_samples)

    # Shuffle the indices
    np.random.shuffle(indices)

    # Use the shuffled indices to shuffle the datasets
    X_train = X_train[indices]
    y_train = y_train[indices]

    X_train_partitions = []
    y_train_partitions = []

    partition_size = int(len(X_train) / partitions)

    for i in range(partitions):
        if i == partitions - 1:
            X_train_partitions.append(X_train[i * partition_size :])
            y_train_partitions.append(y_train[i * partition_size :])
        else:
            X_train_partitions.append(
                X_train[i * partition_size : (i + 1) * partition_size]
            )
            y_train_partitions.append(
                y_train[i * partition_size : (i + 1) * partition_size]
            )

    return X_train_partitions, y_train_partitions


In [5]:
def create_model():
    # network parameters
    hidden_units = 256
    dropout = 0.45
    input_size = 784
    num_labels = 10
    # model is a 3-layer MLP with ReLU and dropout after each layer
    model = Sequential()
    model.add(Dense(hidden_units, input_dim=input_size))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(hidden_units))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(num_labels))
    model.add(Activation("softmax"))
    return model

In [6]:
X_train, y_train = get_train_data()


In [7]:
model = create_model()
rain = Rain(config, model)

2023-07-03 21:02:16,445 [DEBUG] [Rain] Rain is initialized
2023-07-03 21:02:16,446 [DEBUG] [Provisioner] Creating coordinator
2023-07-03 21:02:16,448 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../../Rain/data/coord/
2023-07-03 21:02:16,452 [DEBUG] [Coordinator Ambassador] Coordinator Ambassador is initialized
2023-07-03 21:02:16,455 [DEBUG] [Coordinator] Coordinator is initialized
2023-07-03 21:02:16,457 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../../Rain/data/coord/
2023-07-03 21:02:16,459 [DEBUG] [LocalProvisioner] LocalProvisioner is initialized
2023-07-03 21:02:16,460 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../../Rain/data/divider/
2023-07-03 21:02:16,461 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../../Rain/data/divider/
2023-07-03 21:02:16,462 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../../Rain/data/divider/


In [ ]:
model = rain.train(X_train, y_train, strategy='async')

2023-07-03 21:02:16,476 [DEBUG] [Rain] Creating workers
2023-07-03 21:02:16,489 [INFO] [Provisioner] provisioner is serving
2023-07-03 21:02:16,491 [DEBUG] [Provisioner] Starting coordinator
2023-07-03 21:02:16,494 [INFO] [Coordinator] coordinator is serving
2023-07-03 21:02:16,495 [DEBUG] [Coordinator] sending the num of workers to the provisioner
2023-07-03 21:02:16,507 [DEBUG] [Provisioner] Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
2023-07-03 21:02:16,509 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
2023-07-03 21:02:16,512 [DEBUG] [LocalProvisioner] Creating 3 workers
2023-07-03 21:02:16,513 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../../Rain/data/worker/
2023-07-03 21:02:16,517 [INFO] [Worker_50151] Worker is running on port: 50151
2023-07-03 21:02:16,519 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../../Rain/data/worker/
2023-07-03 21:02:16,523 [INFO] [W

type of the data <class 'numpy.float32'>
type of the data <class 'numpy.float32'>
type of the data <class 'numpy.float32'>


KeyboardInterrupt: 

2023-07-03 21:03:23,231 [DEBUG] [DividerAmbassador] divider is sending x train to the worker
2023-07-03 21:03:23,232 [DEBUG] [DividerAmbassador] divider is sending x train to the worker
2023-07-03 21:03:23,271 [DEBUG] [DividerAmbassador] divider is sending x train to the worker


type of the datatype of the data <class 'numpy.float32'>
 <class 'numpy.float32'>
type of the data <class 'numpy.float32'>


2023-07-03 21:03:23,988 [DEBUG] [DividerAmbassador] divider is sending y train to the worker
2023-07-03 21:03:23,989 [DEBUG] [DividerAmbassador] divider is sending y train to the worker
2023-07-03 21:03:23,990 [DEBUG] [DividerAmbassador] Sending ../../../../Rain/data/divider/3.pkl to worker3
2023-07-03 21:03:23,992 [DEBUG] [DividerAmbassador] Sending ../../../../Rain/data/divider/1.pkl to worker1
2023-07-03 21:03:24,017 [DEBUG] [DividerAmbassador] divider is sending y train to the worker
2023-07-03 21:03:24,020 [DEBUG] [DividerAmbassador] Sending ../../../../Rain/data/divider/2.pkl to worker2
2023-07-03 21:03:25,060 [DEBUG] [DividerAmbassador] divider received: File downloaded successfully after sending the model to worker 1
2023-07-03 21:03:25,063 [DEBUG] [DividerAmbassador] divider received: File downloaded successfully after sending the model to worker 3
2023-07-03 21:03:25,063 [DEBUG] [DividerAmbassador] divider begins executing iteration1 for worker1
2023-07-03 21:03:25,068 [DEBUG

In [ ]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

RuntimeError: You must compile your model before training/testing. Use `model.compile(optimizer, loss)`.

In [ ]:
model = rain.train(X_train, y_train, strategy='sync')

2023-07-03 20:28:25,886 [DEBUG] [Rain] Creating workers
2023-07-03 20:28:25,891 [INFO] [Provisioner] provisioner is serving
2023-07-03 20:28:25,892 [DEBUG] [Provisioner] Starting coordinator
2023-07-03 20:28:25,895 [INFO] [Coordinator] coordinator is serving
2023-07-03 20:28:25,981 [DEBUG] [Coordinator] sending the num of workers to the provisioner
2023-07-03 20:28:25,987 [DEBUG] [Provisioner] Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
2023-07-03 20:28:25,991 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
2023-07-03 20:28:25,995 [DEBUG] [LocalProvisioner] Creating 3 workers
2023-07-03 20:28:26,005 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../../Rain/data/worker/
2023-07-03 20:28:26,033 [INFO] [Worker_50151] Worker is running on port: 50151
2023-07-03 20:28:26,033 [INFO] [Worker_50151] Worker is running on port: 50151
2023-07-03 20:28:26,041 [DEBUG] [TemporaryFilesManager] Creating temp

Epoch 1/2
Epoch 1/2


2023-07-03 20:29:34.838693: W tensorflow/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 62720000 exceeds 10% of free system memory.
2023-07-03 20:29:35.002854: W tensorflow/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 62720000 exceeds 10% of free system memory.


Epoch 1/2
157/157 [==============================] - 3s 11ms/step - loss: 0.1556 - accuracy: 0.9542
Epoch 2/2
157/157 [==============================] - 3s 12ms/step - loss: 0.1524 - accuracy: 0.9545
Epoch 2/2
157/157 [==============================] - 3s 13ms/step - loss: 0.1500 - accuracy: 0.9560
Epoch 2/2
157/157 [==============================] - 2s 13ms/step - loss: 0.1341 - accuracy: 0.9577
sending data to coordinator
157/157 [==============================] - 2s 12ms/step - loss: 0.1324 - accuracy: 0.9593
sending data to coordinator
157/157 [==============================] - 2s 11ms/step - loss: 0.1317 - accuracy: 0.9600
sending data to coordinator


2023-07-03 20:29:40,741 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 1
2023-07-03 20:29:40,744 [DEBUG] [DividerAmbassador] divider begins downloading ../../../../Rain/data/divider/1_1_trained.pkl from worker1
2023-07-03 20:29:41,037 [DEBUG] [DividerAmbassador] Downloaded ../../../../Rain/data/divider/1_1_trained.pkl from worker1 successfully
2023-07-03 20:29:41,130 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 3
2023-07-03 20:29:41,132 [DEBUG] [DividerAmbassador] divider begins downloading ../../../../Rain/data/divider/3_1_trained.pkl from worker3
2023-07-03 20:29:41,339 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 2
2023-07-03 20:29:41,341 [DEBUG] [DividerAmbassador] divider begins downloading ../../../../Rain/data/divider/2_1_trained.pkl from worker2
2023-07-03 20:29:41,368 [DEBUG] [DividerAmbassador] Downloaded ../../../../Rain/data/divider/3_1_

...

2023-07-03 20:30:01,908 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 2
2023-07-03 20:30:01,911 [DEBUG] [DividerAmbassador] divider begins downloading ../../../../Rain/data/divider/2_2_trained.pkl from worker2
2023-07-03 20:30:01,930 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 3
2023-07-03 20:30:01,932 [DEBUG] [DividerAmbassador] divider begins downloading ../../../../Rain/data/divider/3_2_trained.pkl from worker3
2023-07-03 20:30:02,034 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 1
2023-07-03 20:30:02,038 [DEBUG] [DividerAmbassador] divider begins downloading ../../../../Rain/data/divider/1_2_trained.pkl from worker1
2023-07-03 20:30:02,884 [DEBUG] [DividerAmbassador] Downloaded ../../../../Rain/data/divider/2_2_trained.pkl from worker2 successfully
2023-07-03 20:30:02,911 [DEBUG] [DividerAmbassador] Downloaded ../../../../Rain/data/divider/3_2_

Epoch 1/2
Epoch 1/2
Epoch 1/2
157/157 [==============================] - 4s 17ms/step - loss: 0.1178 - accuracy: 0.9642
Epoch 2/2
157/157 [==============================] - 4s 18ms/step - loss: 0.1204 - accuracy: 0.9635
Epoch 2/2
Epoch 2/2
157/157 [==============================] - 3s 17ms/step - loss: 0.1030 - accuracy: 0.9687
sending data to coordinator
sending data to coordinator


2023-07-03 20:30:19,276 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 2
2023-07-03 20:30:19,278 [DEBUG] [DividerAmbassador] divider begins downloading ../../../../Rain/data/divider/2_3_trained.pkl from worker2
2023-07-03 20:30:19,288 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 3
2023-07-03 20:30:19,290 [DEBUG] [DividerAmbassador] divider begins downloading ../../../../Rain/data/divider/3_3_trained.pkl from worker3
2023-07-03 20:30:19,951 [DEBUG] [DividerAmbassador] Downloaded ../../../../Rain/data/divider/3_3_trained.pkl from worker3 successfully
2023-07-03 20:30:19,956 [DEBUG] [DividerAmbassador] Downloaded ../../../../Rain/data/divider/2_3_trained.pkl from worker2 successfully


sending data to coordinator


2023-07-03 20:30:23,786 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 1
2023-07-03 20:30:23,787 [DEBUG] [DividerAmbassador] divider begins downloading ../../../../Rain/data/divider/1_3_trained.pkl from worker1
2023-07-03 20:30:24,067 [DEBUG] [DividerAmbassador] Downloaded ../../../../Rain/data/divider/1_3_trained.pkl from worker1 successfully
2023-07-03 20:30:24,114 [DEBUG] [DeepLearning] Iteration 3/3 complete.
2023-07-03 20:30:24,117 [DEBUG] [DividerAmbassador] divider ambassador stopped serving
2023-07-03 20:30:24,120 [DEBUG] [Divider] Divider stopped serving
2023-07-03 20:30:24,122 [DEBUG] [DividerAmbassador] divider ambassador stopped serving
2023-07-03 20:30:24,123 [DEBUG] [Divider] Divider stopped serving
2023-07-03 20:30:24,125 [INFO] [Provisioner] provisioner stopped serving


In [ ]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 1s 4ms/step - loss: 0.0744 - accuracy: 0.9764

Test accuracy: 97.6%
